# OpenAlex analysis

In [2]:
import pandas as pd

In [3]:
# AltairSaver = altair_save_utils.AltairSaver()

In [4]:
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import utils
import importlib
importlib.reload(utils);

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

2024-05-29 16:24:41,104 - botocore.credentials - INFO - Found credentials in environment variables.
2024-05-29 16:24:42,244 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [87]:
# Labelled data
data_df = utils.load_openalex_data().query("topics != 'arts'")

In [77]:
len(data_df)

23202

In [78]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [79]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

## Baseline trends

Baseline trends for publication counts

In [80]:
baseline_df = utils.get_baseline_openalex()

In [81]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,10068264.6,-3.72067


In [82]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(counts = lambda df: df.counts/1e+6),
    ["Total"],
    variable= "counts",
    variable_title = "Publications (millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [83]:
ts_counts = utils.get_timeseries(data_df, column='id')

In [84]:
ts_counts

,year,counts
0,2013,1912
1,2014,2039
2,2015,2081
3,2016,2461
4,2017,2533
5,2018,2730
6,2019,2669
7,2020,2090
8,2021,2009
9,2022,1750


In [85]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,1889.2,-40.910237


In [86]:
fig = pu.ts_smooth(
    ts_counts.assign(Total="Total"),
    ["Total"],
    variable= "counts",
    variable_title = "Publications",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

Funding for the overall early-years development related research has increased by about 19% in the past five years, which is a positive trend compared to baseline funding which slightly decreased by about 5% in the same time period.

In [15]:
utils.get_data_distribution(data_exploded_df.query("year >= 2019"), column='type', values=['id'])

,type,counts,counts_prop
0,Biosciences,1498,0.081
1,Child care & preschool,4786,0.26
2,Development & learning,7360,0.4
3,General,8968,0.488
4,Health,6591,0.358
5,Parenting,915,0.05
6,Social,6600,0.359
7,Technology,1123,0.061


In [16]:
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='id')

,magnitude,growth,type,counts
7,224.6,84.210526,Technology,1123
1,957.2,25.124378,Child care & preschool,4786
6,1320.0,12.384680,Social,6600
2,1472.0,10.935571,Development & learning,7360
5,183.0,3.738318,Parenting,915
3,1793.6,-11.883217,General,8968
4,1318.2,-19.869565,Health,6591
0,299.6,-29.703833,Biosciences,1498


In [17]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "counts",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [18]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [19]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [20]:
# ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech, 'counts')

alt.Chart(...)

In [21]:
au.ts_magnitude_growth_(ts_counts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
counts,224.6,84.210526


### Distribution of different technologies

In [22]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [23]:
# Total tech funding
counts_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").id.nunique()

In [24]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
    .assign(counts_prop = lambda df: df.counts/counts_total)
)

tech_subtype_dist

,subtype,counts,counts_prop
0,AI,127,0.113090
1,Immersive tech,178,0.158504
2,Internet,610,0.543188
3,Mobile,341,0.303651


### Growth of technology topics

In [116]:
column = 'subtype'
value = 'counts'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,25.4,180.000000,AI
0,35.6,94.202899,Immersive tech
0,122.0,137.696335,Internet
0,68.2,16.939891,Mobile


In [117]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [118]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

### Application distribution

In [119]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)


In [120]:
tech_applications_df

,type,counts,counts_prop
0,Biosciences,53,0.047
1,Child care & preschool,288,0.256
2,Development & learning,436,0.388
3,General,519,0.462
4,Health,270,0.24
5,Parenting,63,0.056
6,Social,288,0.256
7,Technology,1123,1.0


In [121]:
fig = pu.ts_smooth(
    tech_applications_ts,
    tech_applications_ts[column].unique(),
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [122]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')

,magnitude,growth,type,counts
2,87.2,103.144654,Development & learning,436
5,12.6,91.304348,Parenting,63
1,57.6,91.071429,Child care & preschool,288
7,224.6,84.210526,Technology,1123
6,57.6,75.862069,Social,288
3,103.8,55.066079,General,519
4,54.0,24.305556,Health,270
0,10.6,8.571429,Biosciences,53


In [196]:
pd.set_option('display.max_colwidth', 200)
(
    data_exploded_df
    .query('id in @tech_ids')
    # .query("subtype == 'Personal social emotional'")
    .query("type == 'Social'")
    .drop_duplicates(['id'])
    .sort_values('year', ascending=False)
)[['id', 'text', 'topics', 'year']]

,id,text,topics,year
107641,W4320494127,Exploring Caregiver Preferences for Interventions on Child Asthma Exacerbation Risk: An Observational Study (Preprint). <sec> <title>BACKGROUND</title> Maintaining control of asthma symptoms is th...,social_services,2023
82227,W4387938230,"Guidelines for virtual early childhood and family learning: An equity, diversity, inclusion, and decolonization-informed systematic review of the literature. This article presents an equity-inform...",inclusion,2023
80779,W4382515507,Analysis of needs for the development of parent education programs using movies: for fathers with children in early childhood. Objectives This study is a basis for developing a parent education pr...,social_services,2023
81137,W4385468073,Identifying Key Physical and Natural Environmental Correlates of Child Development: An Exploratory Study Using Machine Learning on Data from Pakistan. Bronfenbrenner’s Ecological Systems Theory de...,inequality,2023
81281,W4385843739,THE IMPACT OF PARENTAL MENTAL VIOLENCE ON THE PSYCHOLOGICAL CONDITION OF EARLY CHILDHOOD. Psychological violence has a negative impact on early childhood. Many families cover up violence against y...,inequality,2023
...,...,...,...,...
4037,W20088861,Promoting Excellence within Early Care and Education Providers: A Teacher Education Program Story--RESEARCH. This paper shares the process and experience of a university community partnership to i...,community,2013
4612,W2370328862,Promoting Excellence within Early Care and Education Providers: A Teacher Education. This paper shares the process and experience of a university community partnership to improve the quality of ca...,community,2013
4763,W2479500722,"Insights from Indonesia: Implications for Policy and Practice. No AccessJun 2013Insights from Indonesia: Implications for Policy and PracticeAuthors/Editors: Amer Hasan, Marilou Hyson, Mae Chu Cha...",inequality,2013
4796,W2496891690,Supporting Early Childhood Environmental Education through the Natural Start Alliance.. The Natural Start Alliance is a new initiative of the North American Association for Environmental Education...,community,2013


### Application distribution: More granular subtypes

In [124]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('counts', ascending=False)

,subtype,counts,counts_prop,type
5,Games,273,0.243,General
29,Preschool,261,0.232,Child care & preschool
2,Cognitive development,162,0.144,Development & learning
7,Health,149,0.133,Health
25,Personal social emotional,137,0.122,Development & learning
12,Infancy,133,0.118,General
32,Social services,120,0.107,Social
3,Communication and language,114,0.102,Development & learning
11,Inequalities,92,0.082,Social
4,Community,89,0.079,Social


In [125]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id').sort_values(['type', 'growth'], ascending=False)

,magnitude,growth,subtype,counts,type
4,25.4,180.000000,AI,127,Technology
6,122.0,137.696335,Internet,610,Technology
11,35.6,94.202899,Immersive tech,178,Technology
27,68.2,16.939891,Mobile,341,Technology
3,5.2,185.714286,Labour market,26,Social
10,14.0,108.333333,Inclusion,70,Social
14,24.0,85.416667,Social services,120,Social
21,3.8,62.500000,Income,19,Social
23,18.4,29.166667,Inequalities,92,Social
24,17.8,28.260870,Community,89,Social


In [128]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [172]:
fig = pu.ts_smooth(
    tech_applications_ts,
    cats,
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

### Which technology is applied to most to subtype X?

## Insight 3: Geographical insights

- Top countries in terms of counts
- UK vs baseline growth for overall counts, in technology counts and application counts

In [30]:
data_countries_df = data_exploded_df.explode('country_code').drop_duplicates(['id', 'country_code'])

n_total = data_countries_df.id.nunique()
n_without_country = data_countries_df.country_code.isnull().sum()

print(f"Number of items without country: {n_without_country} ({n_without_country/n_total:.2%})")

Number of items without country: 10079 (26.46%)


## Overall trends

In [51]:
importlib.reload(utils);

growth_df, ts_counts = utils.get_geographical_distribution(
    data_exploded_df
)

(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(15)
)

,magnitude,growth,country_code
0,680.2,-32.864024,US
59,301.8,129.257642,ID
7,186.8,-15.495208,AU
1,145.0,-26.022305,CA
8,104.4,-42.677824,GB
14,75.6,-22.569444,DE
20,75.2,56.521739,CN
4,62.0,-11.055276,SE
35,59.8,53.846154,TR
34,55.4,-1.863354,BR


In [41]:
countries = ['US', 'GB', 'ID']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

### Technology publications

In [52]:
growth_df, ts_counts = utils.get_geographical_distribution(
    data_exploded_df.query('subtype in @tech_subtypes')
)

In [53]:
(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(10)
)

,magnitude,growth,country_code
22,35.0,677.777778,ID
0,31.4,11.235955,US
3,11.6,2.564103,AU
15,5.6,425.000000,HK
2,5.4,5.882353,GB
16,5.0,142.857143,TR
7,4.6,-6.250000,CA
21,4.4,33.333333,ES
1,4.0,-35.294118,CN
31,4.0,800.000000,MY


In [44]:
countries = ['US', 'GB']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

In [66]:
data_countries_gb_df = (
    data_exploded_df
    .explode('country_code')
    .dropna(subset=['country_code'])
    # .query('subtype in @tech_subtypes')
    .drop_duplicates(['id']) 
    .query("country_code == 'GB'")   
)
len(data_countries_gb_df)

1394

In [63]:
data_countries_gb_df.groupby('subtype').agg(counts=('id', 'nunique')).reset_index()

,subtype,counts
0,AI,2
1,Child protection,45
2,Cognitive development,35
3,Communication and language,32
4,Community,16
5,Games,32
6,Genetics,28
7,Health,139
8,Inclusion,11
9,Income,33


In [59]:
pd.set_option('display.max_colwidth', 200)
data_countries_gb_df[['id', 'text']]

,id,text
844,W1995052196,Playful and creative ICT pedagogical framing: a nursery school case study. AbstractThis article reports on the findings of a one-year qualitative study in which a nursery school used information a...
2714,W1779157682,Early Childhood Studies : A social science perspective. Early Childhood Studies: A Social Science Perspective explores key issues in early childhood studies from a variety of social science discip...
5438,W2105785021,"New directions for early literacy in a digital age: The iPad. In this paper, we discuss how iPads offer innovative opportunities for early literacy learning but also present challenges for teacher..."
13703,W2207589504,"New technologies, old dilemmas: theoretical and practical challenges in preschool immersion playrooms. This paper describes some of the findings emerging from a small-scale pilot study investigati..."
18746,W2528971282,‘It’s more funner than doing work’: children’s perspectives on using tablet computers in the early years of school. There is a clamour of voices around the contemporary issue of tablet computers i...
18798,W2407084755,A threat to childhood innocence or the future of learning? Parents’ perspectives on the use of touch-screen technology by 0–3 year-olds in the UK. The rise in personal ownership of touch-screen te...
19168,W1714521979,Lessons from using iPads to understand young children’s creativity. This article explores the use of iPads as part of a child-centred data collection approach to understand young children’s creati...
27559,W2779125856,Supporting the development of young children’s metacognition through the use of video-stimulated reflective dialogue. This paper reports on a study exploring metacognition in young children. Devel...
28498,W2767392963,Early life cognitive function and health behaviours in late childhood: testing the neuroselection hypothesis. Background Higher cognitive function in childhood is associated with healthier behavio...
29221,W2766529925,"Move on up!. Nursery WorldVol. 2017, No. 20 Enabling EnvironmentsMove on up!Carol ArcherCarol ArcherSearch for more papers by this authorCarol ArcherPublished Online:16 Oct 2017https://doi.org/10...."
